# k-Nearest Neighbours, Support Vector Machines and Naive Bayes – Titanic

**Dataset:** Titanic – 891 passengers (Kaggle), the classification example of the slides. File: `../00_Data/titanic.csv`.

**Question:** Can the survival of a Titanic passenger be predicted from a few characteristics with distance-based
methods (k-NN, SVM) and with Naive Bayes?

**Slides:** Part 04 «Classification models» – k-NN, Naive Bayes, SVM.

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to categorical data.

## Concept

The concepts behind this exercise (formulas, worked examples of the slides and additional explanations) are
explained in the notebook [knn_naive_bayes_svm_concept.ipynb](knn_naive_bayes_svm_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

**a) k-NN with and without scaling.** A new passenger is 30 years old and paid a fare of 40 pounds. Five known
passengers:

| passenger | survived? | age [years] | fare [£] |
|---|---|---|---|
| P1 | no | 58 | 41 |
| P2 | yes | 29 | 70 |
| P3 | yes | 33 | 75 |
| P4 | no | 62 | 38 |
| P5 | no | 31 | 8 |

1. Compute the Euclidean distances on the **original scale** and classify the new passenger with $k=3$.
2. Standardise the differences with the standard deviations of the whole data set (age: 13 years, fare: 50 £), i.e.
   use $\Delta z = \Delta x / s$, and classify again with $k=3$.
3. Why do the results differ?

**b) Naive Bayes.** Of the 891 passengers, 342 survived and 549 died. Among the survivors there were 233 women and 136
first-class passengers; among those who died there were 81 women and 80 first-class passengers. Compute the Naive Bayes
scores for a **woman in 1st class** (as on the slide «Naive Bayes classifier») and her probability of survival. Compare
with the observed survival rate of women in 1st class (91 of 94).

**c)** Verify your results with Python.

In [ ]:
# a) k-NN with and without scaling
new = np.array([30, 40])
X_known = np.array([[58, 41], [29, 70], [33, 75], [62, 38], [31, 8]])
survived = np.array(['no', 'yes', 'yes', 'no', 'no'])
sd = np.array([13, 50])

d_raw = np.sqrt(((X_known - new)**2).sum(axis=1))
d_std = np.sqrt((((X_known - new) / sd)**2).sum(axis=1))
print(pd.DataFrame({'survived': survived, 'distance (raw)': d_raw.round(1),
                    'distance (standardised)': d_std.round(3)}, index=['P1', 'P2', 'P3', 'P4', 'P5']))
for name, d in [('raw', d_raw), ('standardised', d_std)]:
    print(f'{name:13s}: 3 nearest = {survived[np.argsort(d)[:3]].tolist()}')

# b) Naive Bayes
score_surv = 342/891 * 233/342 * 136/342
score_died = 549/891 * 81/549 * 80/549
print(f'\nscore(survived) = {score_surv:.4f}, score(died) = {score_died:.4f}')
print(f'P(survived | woman, 1st class) = {score_surv / (score_surv + score_died):.3f}')
print(f'Observed survival rate of women in 1st class = {91/94:.3f}')

**Solution:**

a) 1. Raw distances: P1 ≈ 28.0, P2 ≈ 30.0, P3 ≈ 35.1, P4 ≈ 32.1, P5 ≈ 32.0 → 3 nearest: P1, P2, P5 → **died** (2 of 3).

2. Standardised: P1 = $\sqrt{(28/13)^2 + (1/50)^2} \approx 2.154$; P2 = $\sqrt{(1/13)^2+(30/50)^2} \approx 0.605$;
P3 = $\sqrt{(3/13)^2+(35/50)^2} \approx 0.737$; P4 ≈ 2.462; P5 = $\sqrt{(1/13)^2+(32/50)^2} \approx 0.645$ →
3 nearest: P2, P5, P3 → **survived** (2 of 3).

3. On the original scale, one year of age counts as much as one pound of fare. Because ages differ less than fares
(standard deviation 13 vs. 50), a 28-year age difference (P1) looks "close". After standardising, both features count
in units of their typical spread, and P1 (58 years old) is far away.

b) score(survived) = $\frac{342}{891}\cdot\frac{233}{342}\cdot\frac{136}{342} \approx 0.104$;
score(died) = $\frac{549}{891}\cdot\frac{81}{549}\cdot\frac{80}{549} \approx 0.0132$ → predict **survived**;
P(survived | woman, 1st class) $= 0.104/(0.104+0.0132) \approx 0.89$. The observed survival rate is even higher (91/94
≈ 0.97): sex and class are not independent given survival (almost all women in 1st class survived) – the "naive"
assumption leads to a biased probability, but the predicted class is correct.

## Exercise 2: Prepare the Titanic data

a) Import the Titanic data and prepare the features `male`, `Age` (missing values replaced by the median), `Pclass`,
`SibSp`, `Parch` and `Fare`. Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

b) Create a scatter plot of `Age` against `Fare` (logarithmic y-axis), coloured by `Survived`, and compare the scales of
the features.

In [ ]:
# a) Import and prepare the data
titanic = pd.read_csv('../00_Data/titanic.csv', sep=',')
titanic['male'] = (titanic['Sex'] == 'male').astype(int)
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
features = ['male', 'Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
X, y = titanic[features], titanic['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)
print(X_train.describe().round(1))

# b) Scatter plot Age vs. Fare
plt.figure(figsize=(8, 4.5))
sns.scatterplot(data=titanic, x='Age', y=titanic['Fare'] + 1, hue='Survived', s=18, alpha=0.7)
plt.yscale('log')
plt.ylabel('Fare + 1 [£] (log scale)')
plt.title('Titanic passengers')
plt.show()

**Solution:**

The features are on very different scales: `Fare` ranges from 0 to 512 £ (standard deviation ≈ 50), `Age` from 0 to 80
years (≈ 13), `male` is 0/1 and `Pclass` 1–3. Survivors are more frequent among passengers with high fares and among
children; in the age–fare plane the two classes overlap strongly – sex, the most important feature, is not visible in
this plot.

## Exercise 3: k-NN – the effect of scaling and the choice of k

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`. Explain the difference.

b) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$. Choose $k$ and
compute accuracy and confusion matrix on the test set.

c) Predict the survival probability of a 30-year-old man in 3rd class without relatives (fare 8 £) and of a
30-year-old woman in 1st class without relatives (fare 80 £).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Unscaled vs. scaled
for name, m in [('unscaled', KNeighborsClassifier(n_neighbors=5)),
                ('scaled', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)))]:
    acc = cross_val_score(m, X_train, y_train, cv=cv, scoring='accuracy')
    print(f'k-NN ({name:8s}): CV accuracy = {acc.mean():.3f} (+/- {acc.std():.3f})')

# b) Choice of k
ks = range(1, 41)
cv_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                          X_train, y_train, cv=cv).mean() for k in ks]
plt.figure(figsize=(7, 3.5))
plt.plot(ks, cv_acc, marker='o', markersize=4)
plt.xlabel('k (number of neighbours)')
plt.ylabel('5-fold CV accuracy')
plt.grid(alpha=0.3)
plt.show()

best_k = ks[int(np.argmax(cv_acc))]
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_train, y_train)
print(f'Chosen k = {best_k}, CV accuracy = {max(cv_acc):.3f}, test accuracy = {knn.score(X_test, y_test):.3f}')
ConfusionMatrixDisplay.from_estimator(knn, X_test, y_test, display_labels=['died', 'survived'],
                                      cmap='Blues', colorbar=False)
plt.title(f'k-NN (k = {best_k}) on the test set')
plt.show()

# c) Two new passengers
new_passengers = pd.DataFrame({'male': [1, 0], 'Age': [30, 30], 'Pclass': [3, 1],
                               'SibSp': [0, 0], 'Parch': [0, 0], 'Fare': [8, 80]})
print(pd.DataFrame(knn.predict_proba(new_passengers), columns=['P(died)', 'P(survived)'],
                   index=['man, 3rd class', 'woman, 1st class']).round(2))

**Solution:**

a) Without scaling the CV accuracy is only about **0.71**, with scaling about **0.82**. Without scaling, the distance is
dominated by `Fare` (differences of tens or hundreds of pounds) and `Age`; the most important feature `male` (0/1) and the
class (1–3) hardly contribute to the distance – exactly the effect of Exercise 1a. The scaler must be part of the
pipeline so that it is fitted on the training folds only (no data leakage).

b) The CV accuracy varies only between about 0.80 and 0.83 for most $k$; CV selects $k = 3$ (CV accuracy ≈ 0.83, test
accuracy ≈ 0.80). Small $k$ give flexible, larger $k$ smoother decision boundaries.

c) With $k = 3$ all three neighbours of the 30-year-old man in 3rd class died, and all three neighbours of the woman in
1st class survived → probabilities of 0 and 1. With small $k$, k-NN probabilities are very coarse (only 0, 1/3, 2/3, 1);
a larger $k$ gives finer probabilities.

## Exercise 4: Support vector machines – C and kernel

a) Compute the 5-fold CV accuracy and the number of support vectors of
`make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set (all six features).

b) To *see* the boundaries, fit the same six models on only the two features `Age` and `log_fare = log(1 + Fare)` and
plot the decision regions in a 2 × 3 grid (the code for the grid is given), with the support vectors circled.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
# a) CV accuracy and number of support vectors (all features)
rows = []
for kernel in ['linear', 'rbf']:
    for C in [0.01, 1, 100]:
        svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C))
        acc = cross_val_score(svm, X_train, y_train, cv=cv).mean()
        n_sv = len(svm.fit(X_train, y_train)[-1].support_)
        rows.append({'kernel': kernel, 'C': C, 'support vectors': n_sv, 'CV accuracy': round(acc, 3)})
print(pd.DataFrame(rows).to_string(index=False))

# b) Decision regions with two features
X2_train = pd.DataFrame({'Age': X_train['Age'], 'log_fare': np.log1p(X_train['Fare'])})
xx, yy = np.meshgrid(np.linspace(0, 82, 300), np.linspace(0, 6.5, 300))
grid = pd.DataFrame({'Age': xx.ravel(), 'log_fare': yy.ravel()})

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
for row, kernel in enumerate(['linear', 'rbf']):
    for col, C in enumerate([0.01, 1, 100]):
        ax = axes[row, col]
        svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C)).fit(X2_train, y_train)
        zz = svm.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['tab:red', 'tab:blue'], alpha=0.15)
        for cls, color, label in [(0, 'tab:red', 'died'), (1, 'tab:blue', 'survived')]:
            m = y_train.values == cls
            ax.scatter(X2_train['Age'][m], X2_train['log_fare'][m], s=8, color=color, label=label)
        sv = X2_train.iloc[svm[-1].support_]
        ax.scatter(sv['Age'], sv['log_fare'], s=40, facecolors='none', edgecolors='black', linewidths=0.5,
                   label='support vectors')
        ax.set_title(f'kernel = {kernel}, C = {C}: {len(sv)} support vectors')
        ax.set_xlabel('Age [years]' if row == 1 else '')
        ax.set_ylabel('log(1 + fare)' if col == 0 else '')
axes[0, 0].legend(loc='upper right', fontsize=8)
plt.show()

**Solution:**

a) and b):
- **Small $C$ (0.01):** very wide margin, many training errors are tolerated → almost all points are support vectors. With
  the RBF kernel the model predicts "died" for everybody (CV accuracy ≈ 0.62 = share of non-survivors, 547 of 712 points
  are support vectors) → **underfitting**.
- **$C = 1$:** the RBF SVM reaches the best CV accuracy (≈ 0.82, similar to k-NN); the linear SVM ≈ 0.79.
- **Large $C$ (100):** narrow margin, the model tries to classify every training point correctly. With the RBF kernel the
  boundary in the age–fare plot becomes very wiggly ("islands" around single passengers) and the CV accuracy decreases
  slightly (≈ 0.82) → beginning **overfitting**.
- **Kernel:** the linear kernel gives straight boundaries – for the linear SVM, $C$ hardly matters here (≈ 0.79 for all
  values). The RBF kernel can model interactions (e.g. children survive more often regardless of the fare), which gives
  better results.

c) An RBF SVM with $C = 1$. $C$ (and the kernel width `gamma`) should be tuned with cross-validation. The two-feature plot
is only for illustration – without the feature `male` the classes overlap strongly.

## Exercise 5: Naive Bayes with categorical features

a) Fit a `CategoricalNB(alpha=1e-10)` (practically no smoothing) on the two features `male` and `Pclass` of **all**
passengers and predict the survival probability of a woman in 1st class. Compare with your hand calculation of
Exercise 1b.

b) Compute the 5-fold CV accuracy of a `CategoricalNB()` with the features `male`, `Pclass` and `age_group`
(0 = child under 10, 1 = 10–59 years, 2 = 60 years and older) on the training set. Compare with k-NN and SVM.

In [ ]:
# a) CategoricalNB on male and Pclass (all passengers)
X_cat = pd.DataFrame({'male': titanic['male'], 'Pclass': titanic['Pclass'] - 1})   # categories must start at 0
nb_cat = CategoricalNB(alpha=1e-10).fit(X_cat, titanic['Survived'])
woman_1st = pd.DataFrame({'male': [0], 'Pclass': [0]})
print('P(survived | woman, 1st class) =', round(nb_cat.predict_proba(woman_1st)[0, 1], 3))

# b) CategoricalNB with age group
def categorical_features(X):
    return pd.DataFrame({'male': X['male'], 'Pclass': X['Pclass'] - 1,
                         'age_group': pd.cut(X['Age'], [-1, 9.99, 59.99, 100], labels=False)})

acc_nb = cross_val_score(CategoricalNB(), categorical_features(X_train), y_train, cv=cv).mean()
print(f'CategoricalNB (male, Pclass, age_group): CV accuracy = {acc_nb:.3f}')

**Solution:**

a) `CategoricalNB` gives P(survived | woman, 1st class) ≈ **0.887** – exactly the hand calculation of Exercise 1b.

b) With sex, class and age group, Naive Bayes reaches a CV accuracy of about **0.79** – a little lower than the scaled
k-NN and the RBF SVM (≈ 0.82–0.83), although it uses only three coarse categorical features and is extremely simple and
fast. The independence assumption is violated (e.g. sex and class interact), which costs some accuracy.

## Exercise 6: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

**Solution:**

a) **k-NN and SVM** are based on distances / inner products → scale the features. Naive Bayes estimates one distribution
per feature and class, scaling is not needed.

b) **k-NN** has practically no training ("lazy learner": it only stores the data), but for each prediction the distances
to all training points must be computed.

c) **Naive Bayes** (one parameter per word and class) and linear **SVMs** (widely used for text classification) cope well
with many features. For k-NN, distances in very high-dimensional spaces become less informative ("curse of
dimensionality").

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')